# Full GUDS OOD on PAD-UFES-20 (NVIDIA-layout)

The notebook trains a source-only control and a fixed target-assisted OOD model. The latter uses **only** PAD `imgs_part_1/2` as unlabeled outlier exposure and reserves `imgs_part_3` for the final OOD test. It is therefore a target-assisted domain-shift result, not a clean unseen-domain OOD claim. No configuration is selected using `imgs_part_3`.

In [ ]:
# Cell 1 -- pull code; use attached Kaggle inputs first, KaggleHub second
import os, sys, json, shutil, subprocess, shlex
from pathlib import Path

REPO_URL = 'https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/kaggle/working/MDEP-Microglial-Driven-Evidential-Pruning')
WORK, OUT = Path('/kaggle/working'), Path('/kaggle/working/paper_experiment_outputs')
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print('\n$', shlex.join(cmd), flush=True)
    merged = os.environ.copy(); merged.update({str(k): str(v) for k, v in (env or {}).items()})
    subprocess.run(cmd, cwd=str(cwd or REPO_ROOT), env=merged, check=True)

if not (REPO_ROOT / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_ROOT], cwd=WORK)
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)

def find_isic(root):
    root = Path(root)
    for csv in [root / 'train-metadata.csv', *root.rglob('train-metadata.csv')]:
        p = csv.parent
        if (p / 'train-image.hdf5').exists() or (p / 'train-image' / 'image').exists(): return p
    return None

def find_pad(root):
    root = Path(root)
    for csv in [root / 'metadata.csv', *root.rglob('metadata.csv')]:
        p = csv.parent
        if all((p / x).exists() for x in ('imgs_part_1', 'imgs_part_2', 'imgs_part_3')): return p
    return None

ISIC_ROOT = next((p for p in (find_isic(x) for x in [Path('/kaggle/input/competitions/isic-2024-challenge'), Path('/kaggle/input/isic-2024-challenge')]) if p), None)
PAD_ROOT = next((p for p in (find_pad(x) for x in [Path('/kaggle/input/datasets/mahdavi1202/skin-cancer'), Path('/kaggle/input/skin-cancer')]) if p), None)
if ISIC_ROOT is None or PAD_ROOT is None:
    try:
        import kagglehub
        if ISIC_ROOT is None: ISIC_ROOT = find_isic(kagglehub.competition_download('isic-2024-challenge'))
        if PAD_ROOT is None: PAD_ROOT = find_pad(kagglehub.dataset_download('mahdavi1202/skin-cancer'))
    except Exception as exc:
        raise RuntimeError('Attach ISIC-2024 and PAD-UFES-20 through Add Data (or enable Internet and accept the ISIC competition rules).') from exc
assert ISIC_ROOT is not None, 'ISIC train-metadata/train images not found'
assert PAD_ROOT is not None, 'PAD metadata plus imgs_part_1/2/3 not found'
PAD_META = PAD_ROOT / 'metadata.csv'
assert PAD_META.exists()
os.environ['ISIC_ROOT'] = str(ISIC_ROOT)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
print('[OK] commit =', commit)
print('[OK] ISIC_ROOT =', ISIC_ROOT)
print('[OK] PAD_ROOT =', PAD_ROOT, '| OE=imgs_part_1/2, test=imgs_part_3')


In [ ]:
# Cell 2 -- fixed, leakage-safe OOD protocol. Do not tune on imgs_part_3.
SEEDS = (42, 123, 456)
SPLIT_SEED, EPOCHS, BATCH_SIZE, LR = 42, 40, 32, 4e-5
PROFILE = 'nvidia_v3'
VARIANTS = {
    # Clean external-OOD control: no PAD image is seen while training.
    'source_only': {'suffix': '_ood_padufes_source_only_v1', 'oe': None, 'primary': 'knn_layer3'},
    # Fixed before testing: PAD parts 1/2 only, detached head protects ISIC classifier/topology.
    'target_assisted_projection': {'suffix': '_ood_padufes_projection_v1', 'oe': str(PAD_ROOT), 'primary': 'knn_ood_projection'},
}
OOD_OUT = OUT / 'ood_padufes'; OOD_OUT.mkdir(parents=True, exist_ok=True)

def model_metrics(variant, seed):
    return OUT / 'isic' / f"full_guds{VARIANTS[variant]['suffix']}" / f'seed_{seed}' / 'metrics.json'

def train_complete(variant, seed):
    try:
        d = json.loads(model_metrics(variant, seed).read_text())
        return (d.get('protocol_profile') == PROFILE and d.get('sparsity_layout') == 'nvidia_kcrs' and
                d.get('epochs') == EPOCHS and d.get('model_seed', d.get('seed')) == seed and
                d.get('outlier_exposure') == VARIANTS[variant]['oe'] and
                d.get('ood_loss_target') == 'projection' and d.get('checkpoint_selection', {}).get('metric') == 'last')
    except (OSError, ValueError, TypeError): return False

for variant, cfg in VARIANTS.items():
    for seed in SEEDS:
        if train_complete(variant, seed):
            print(f'[SKIP] train {variant}, seed={seed}')
            continue
        cmd = [sys.executable, '-u', 'experiments/isic_paper_experiments.py', '--experiment', 'full_guds',
               '--seed', str(seed), '--split_seed', str(SPLIT_SEED), '--epochs', str(EPOCHS),
               '--batch_size', str(BATCH_SIZE), '--lr', str(LR), '--subsample_scope', 'train',
               '--subsample_ratio', '20', '--checkpoint_selection', 'last',
               '--structural_proxy_batches', '4', '--protocol_profile', PROFILE,
               '--run_suffix', cfg['suffix'], '--ood_loss_target', 'projection',
               '--lambda_ood', '0.003', '--ood_batch_ratio', '0.125', '--ood_start_epoch', '30']
        if cfg['oe'] is not None:
            cmd += ['--outlier_exposure', cfg['oe']]
        run(cmd, cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})

assert all(train_complete(v, s) for v in VARIANTS for s in SEEDS), 'A train run failed or did not meet the NVIDIA OOD protocol.'
print('[DONE] All Full GUDS OOD checkpoints passed the NVIDIA-layout gate.')


In [ ]:
# Cell 3 -- one held-out PAD test per checkpoint, then preserve variant-specific summaries
def audit_path(variant, seed): return OOD_OUT / f'{variant}_seed_{seed}_imgs_part_3.json'

def eval_complete(variant, seed):
    try:
        d = json.loads(audit_path(variant, seed).read_text())
        r = d['external_summary']
        return (d['variant'] == variant and r['checkpoint_protocol']['nvidia_layout_validated'] and
                r['external_dataset']['pad_ufes_partition'] == 'imgs_part_3' and
                r['primary_ood']['score'] == VARIANTS[variant]['primary'])
    except (OSError, KeyError, ValueError, TypeError): return False

for variant, cfg in VARIANTS.items():
    for seed in SEEDS:
        if eval_complete(variant, seed):
            print(f'[SKIP] held-out OOD {variant}, seed={seed}')
            continue
        checkpoint = model_metrics(variant, seed).with_name('model_state.pth')
        run([sys.executable, '-u', 'experiments/run_external_validation.py',
             '--model_path', checkpoint, '--seed', str(seed), '--split_seed', str(SPLIT_SEED),
             '--custom_image_folder', PAD_ROOT, '--pad_ufes_csv', PAD_META,
             '--pad_ufes_partition', 'imgs_part_3', '--knn_primary_layer', 'layer3',
             '--primary_ood_score', cfg['primary'], '--fairness_min_group_size', '20',
             '--fairness_min_class_size', '10'], cwd=REPO_ROOT, env={'ISIC_ROOT': ISIC_ROOT, 'MDEP_DETERMINISTIC': '1'})
        candidates = list((OUT / 'external_validation').glob(f'external_validation_seed_{seed}_imgs_part_3_*.json'))
        assert candidates, f'No external-validation JSON emitted for seed={seed}'
        raw = max(candidates, key=lambda p: p.stat().st_mtime)
        wrapped = {'variant': variant, 'checkpoint': str(checkpoint), 'git_commit': commit,
                   'target_assisted': cfg['oe'] is not None, 'external_summary': json.loads(raw.read_text())}
        audit_path(variant, seed).write_text(json.dumps(wrapped, indent=2))

assert all(eval_complete(v, s) for v in VARIANTS for s in SEEDS), 'OOD evaluation incomplete.'
print('[DONE] PAD imgs_part_3 remained held out and all seed summaries were preserved.')


In [ ]:
# Cell 4 -- report both protocols; no automatic winner selection on the held-out test
import pandas as pd
rows = []
for variant in VARIANTS:
    for seed in SEEDS:
        d = json.loads(audit_path(variant, seed).read_text())['external_summary']
        p = d['primary_ood']['result']
        rows.append({'variant': variant, 'seed': seed, 'target_assisted': VARIANTS[variant]['oe'] is not None,
                     'primary_score': d['primary_ood']['score'], 'full_auroc': p['full_auroc'],
                     'full_aupr': p['full_aupr'], 'balanced_auroc': p['bal_auroc_mean'],
                     'balanced_aupr': p['bal_aupr_mean'], 'fairness_status': d['fairness']['status']})
table = pd.DataFrame(rows)
table.to_csv(OOD_OUT / 'pad_ufes_ood_seed_metrics.csv', index=False)
manifest = {'git_commit': commit, 'profile': PROFILE, 'sparsity_layout': 'nvidia_kcrs', 'seeds': SEEDS,
            'pad_root': str(PAD_ROOT), 'test_partition': 'imgs_part_3',
            'target_assisted_oe_partitions': ['imgs_part_1', 'imgs_part_2'], 'variants': VARIANTS,
            'warning': 'target_assisted_projection is not a clean unseen-domain OOD claim; report it separately from source_only.'}
(OOD_OUT / 'pad_ufes_ood_manifest.json').write_text(json.dumps(manifest, indent=2))
display(table.groupby(['variant', 'target_assisted', 'primary_score'])[['full_auroc', 'full_aupr', 'balanced_auroc', 'balanced_aupr']].agg(['mean', 'std']))
print('[SAVED]', OOD_OUT / 'pad_ufes_ood_seed_metrics.csv')
